# Gemma 4 E2B × TinyCeNN — Integrated Memory V2

Fixes the V1 checkpoint-loading bug for `google/gemma-4-E2B`: text weights are remapped from `model.language_model.*` into `Gemma4ForCausalLM`.

**Default profile: `balanced`.**

Safe TinyCeNN targets:
- conservative: layer `4`
- expanded: layers `4,9`

Shared-KV producer/consumer layers remain native.


In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

REPO = Path(tempfile.mkdtemp(prefix='gemma4-e2b-cenn-v2-')) / 'TinyCeNN-LM'
subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'transformers==5.17.0','huggingface_hub>=0.36.2','datasets>=3,<6',
    'accelerate','pytest','pandas','matplotlib'
], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'], check=True)

os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO),str(REPO/'src')])
sys.path[:0] = [str(REPO),str(REPO/'src')]

import torch
from transformers import AutoConfig
MODEL_ID = 'google/gemma-4-E2B'
cfg = AutoConfig.from_pretrained(MODEL_ID, token=HF_TOKEN).get_text_config(decoder=True)
FULL = [i for i,t in enumerate(cfg.layer_types) if t == 'full_attention']
FIRST_SHARED = cfg.num_hidden_layers - cfg.num_kv_shared_layers
NONSHARED_FULL = [i for i in FULL if i < FIRST_SHARED]
SHARED_KV_PRODUCER = max(NONSHARED_FULL)
SAFE_REPLACEABLE = [i for i in NONSHARED_FULL if i != SHARED_KV_PRODUCER]

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Full attention:', FULL)
print('First shared layer:', FIRST_SHARED)
print('Shared-KV producer:', SHARED_KV_PRODUCER)
print('Safely replaceable:', SAFE_REPLACEABLE)

assert cfg.model_type == 'gemma4_text'
assert FULL == [4,9,14,19,24,29,34]
assert cfg.num_kv_shared_layers == 20
assert SAFE_REPLACEABLE == [4,9]


In [ ]:
PROFILE = 'balanced' # @param ['smoke','balanced','extended']
SAVE_TO_DRIVE = True # @param {type:'boolean'}

PROFILES = {
    'smoke': dict(train_contexts='96,128',test_contexts='96,128,256',block_size=16,features=32,
                  train_documents=4,validation_documents=2,test_documents=2,warm_documents=2,
                  warm_steps=2,joint_steps=4,eval_every=2,timing_documents=1,timing_repeats=1,
                  decode_tokens=8,loss_chunk=2),
    'balanced': dict(train_contexts='128,256',test_contexts='128,256,512,1024',block_size=32,features=64,
                     train_documents=24,validation_documents=6,test_documents=8,warm_documents=4,
                     warm_steps=30,joint_steps=60,eval_every=15,timing_documents=1,timing_repeats=1,
                     decode_tokens=24,loss_chunk=4),
    'extended': dict(train_contexts='128,256,512',test_contexts='128,256,512,1024,2048',block_size=32,features=96,
                     train_documents=48,validation_documents=10,test_documents=16,warm_documents=8,
                     warm_steps=60,joint_steps=120,eval_every=20,timing_documents=2,timing_repeats=2,
                     decode_tokens=32,loss_chunk=4),
}

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime.')

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/TinyCeNN/gemma4-e2b-integrated-v2')
else:
    BASE = Path('/content/gemma4-e2b-integrated-v2')

BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = BASE / run_id
LOG = BASE / (run_id + '.log')
RUN = dict(PROFILES[PROFILE], seed=2041)
print(json.dumps(RUN, indent=2))
print('Results:', OUT)


In [ ]:
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)

env = dict(os.environ, CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
tests = ['tests/test_gemma4_checkpoint.py','tests/test_gemma4_integrated_memory.py']
r = subprocess.run([sys.executable,'-m','pytest','-q',*tests],
                   cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode:
    raise RuntimeError(f'Gemma 4 V2 preflight failed: {r.returncode}')

BENCHMARK = REPO / 'scripts/benchmark_gemma4_e2b_integrated_memory_v2.py'
probe = subprocess.run([sys.executable,str(BENCHMARK),'--help'],
                       cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(probe.stdout.splitlines()[0] if probe.stdout else '')
if probe.returncode:
    raise RuntimeError('Gemma 4 V2 benchmark entrypoint failed')
print('✅ Preflight passed: real text-weight remapping is active')


In [ ]:
cmd = [sys.executable,'-u',str(BENCHMARK),'--base-model',MODEL_ID,'--output-dir',str(OUT)]
for k,v in RUN.items():
    cmd += ['--' + k.replace('_','-'), str(v)]
print('Running:', BENCHMARK.name)
print(' '.join(cmd))

try:
    with LOG.open('w') as log:
        with subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1,env=os.environ.copy()) as p:
            for line in p.stdout:
                print(line,end='',flush=True)
                log.write(line); log.flush()
            status = p.wait()
    if status:
        raise RuntimeError(f'Run failed: {status}; inspect {LOG}')
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG, OUT/'console.log')
        print('Archive:', shutil.make_archive(str(OUT)+'-results','zip',root_dir=OUT))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

s = pd.read_csv(OUT/'integrated_summary.csv')
selected = json.loads((OUT/'selection.json').read_text())['selected']
cols = [c for c in [
    'candidate','context','test_nll','test_perplexity','ppl_ratio','adapted_ppl_ratio',
    'total_cache_ratio','prefill_speedup','decode_speedup','remaining_full_attention_layers',
    'cached_logits_nmse','teacher_cached_logits_nmse','candidate_top1_mismatches',
    'teacher_top1_mismatches','allowed_top1_mismatches',
    'cache_equivalence_top1_floor','cache_equivalence_passed','selected_on_validation'
] if c in s.columns]
print('Locked validation selection:', selected)
display(s[cols].sort_values(['candidate','context']).reset_index(drop=True))

x = s[s.candidate == selected].sort_values('context')
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(x.context,x.ppl_ratio,marker='o',label='PPL ratio')
ax.plot(x.context,x.total_cache_ratio,marker='s',label='cache ratio')
ax.axhline(1,linestyle='--')
ax.set_xscale('log',base=2)
ax.set_xlabel('Context')
ax.legend()
ax.set_title(selected)
plt.show()


In [ ]:
import gc
from transformers import AutoTokenizer
from transformers.cache_utils import DynamicCache
from tinycenn_lm.gemma4_checkpoint import load_gemma4_text_causal
from tinycenn_lm.gemma4_integrated_memory import restore_student, new_cache, inference_mode, native_dtype, text_config

manifest = json.loads((OUT/'manifest.json').read_text())
report = json.loads((OUT/'integrated_report.json').read_text())
SELECTED = json.loads((OUT/'selection.json').read_text())['selected']
record = next(r for r in report['candidates'] if r['candidate'] == SELECTED)
checkpoint = OUT / record['checkpoint']
device = torch.device('cuda')
dtype = native_dtype(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,revision=manifest['model_revision'],token=HF_TOKEN)
original, load_info = load_gemma4_text_causal(
    MODEL_ID, revision=manifest['model_revision'], dtype=dtype,
    attn_implementation='sdpa', token=HF_TOKEN, device=device
)
assert load_info['core_missing_keys'] == []
assert load_info['unexpected_text_keys'] == []

payload = torch.load(checkpoint,map_location='cpu',weights_only=True)
cenn = restore_student(original,payload).to(device).eval()

@torch.no_grad()
def native_generate(model, ids, max_new_tokens=64):
    cache = DynamicCache(config=text_config(model))
    logits = model(input_ids=ids,past_key_values=cache,use_cache=True,logits_to_keep=1).logits[:,-1]
    out=[]
    eos = tokenizer.eos_token_id
    eos = {eos} if isinstance(eos,int) else set(eos or [])
    for _ in range(max_new_tokens):
        tok = logits.argmax(-1,keepdim=True); out.append(tok)
        if int(tok.item()) in eos: break
        logits = model(input_ids=tok,past_key_values=cache,use_cache=True,logits_to_keep=1).logits[:,-1]
    return torch.cat(out,1)

@torch.no_grad()
def cenn_generate(model, ids, max_new_tokens=64):
    cache = new_cache(model)
    out=[]
    eos = tokenizer.eos_token_id
    eos = {eos} if isinstance(eos,int) else set(eos or [])
    with inference_mode(model,str(dtype).replace('torch.','')):
        logits = model(input_ids=ids,past_key_values=cache,use_cache=True,logits_to_keep=1).logits[:,-1]
        for _ in range(max_new_tokens):
            tok = logits.argmax(-1,keepdim=True); out.append(tok)
            if int(tok.item()) in eos: break
            logits = model(input_ids=tok,past_key_values=cache,use_cache=True,logits_to_keep=1).logits[:,-1]
    return torch.cat(out,1)

PROMPTS = [
    'Question: Why is the sky blue?\nAnswer:',
    'Question: Calculate 18 times 7.\nAnswer:',
    'Question: A train travels 120 km in 90 minutes. What is its average speed in km/h?\nAnswer:',
    'Write a short Python function that checks whether an integer is prime.\n',
    'Explain the difference between RAM and SSD storage in three short points.\n',
    'Complete this sentence clearly: Artificial intelligence can help people by',
]

rows=[]
for i,prompt in enumerate(PROMPTS,1):
    ids = tokenizer(prompt,return_tensors='pt',add_special_tokens=True).input_ids.to(device)
    a = native_generate(original,ids,64)
    b = cenn_generate(cenn,ids,64)
    ta = tokenizer.decode(a[0],skip_special_tokens=True).strip()
    tb = tokenizer.decode(b[0],skip_special_tokens=True).strip()
    n = min(a.shape[1],b.shape[1])
    agree = float((a[:,:n] == b[:,:n]).float().mean()) if n else 0.0
    prefix=0
    for x,y in zip(a[0].tolist(),b[0].tolist()):
        if x != y: break
        prefix += 1
    print('\n'+'='*100)
    print(f'TEST {i}: {prompt}')
    print('\nORIGINAL GEMMA 4:\n',ta)
    print('\nTinyCeNN:\n',tb)
    print(f'\nToken agreement: {agree:.1%}; identical prefix: {prefix}')
    rows.append(dict(prompt=prompt,original=ta,cenn=tb,token_agreement=agree,
                     identical_prefix_tokens=prefix,original_tokens=int(a.shape[1]),cenn_tokens=int(b.shape[1])))

chat = pd.DataFrame(rows)
display(chat[['prompt','token_agreement','identical_prefix_tokens','original_tokens','cenn_tokens']])
chat.to_csv(OUT/'chat_comparison.csv',index=False)
(OUT/'chat_comparison.json').write_text(json.dumps(rows,indent=2,ensure_ascii=False))
print('Mean token agreement:', f"{chat.token_agreement.mean():.2%}")

del cenn, original
gc.collect()
torch.cuda.empty_cache()
